In [1]:
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou
import torch
import napari
import os
from skimage import io, draw, measure
import numpy as np
import pandas as pd
import dask.array as da
from ome_zarr.classes import NgffMultiscales
from ome_zarr.scene import NgffScene

from napari_autoscape.models.leica import (
 	StageOverviewRegions,
  	CompoundShape,
  	Point,
  	Vertex,
  	ShapeList,
  	StackList,
  	StackEntry,
  	DefinedRegionEntry,
  	DefinedRegions,
    Regions,
    serialize_to_rgn,
)
from lxml import etree

In [2]:
model = YOLO(r"train8\weights\best.pt")

In [3]:
viewer = napari.Viewer()

In [4]:
root = r'E:\UserData\Johannes\20260420\2026_04_21_09_49_52--TRG1970_001\TileScan 1'

In [5]:
scene = NgffScene.from_ome_zarr(os.path.join(root, r"scene.ome.zarr"))
scene.images

[NgffMultiscales(method=<Methods.RESIZE: 'resize'>)]

In [6]:
def yolo_box2rect(box):
	y_center, x_center, height, width = box
	x_min = int(x_center - width / 2)
	x_max = int(x_center + width / 2)
	y_min = int(y_center - height / 2)
	y_max = int(y_center + height / 2)

	p0 = (x_min, y_min)
	p1 = (x_max, y_min)
	p2 = (x_max, y_max)
	p3 = (x_min, y_max)

	rect = [p0, p1, p2, p3]
	return rect

In [7]:
def sliding_window_inference(image, model, window_size=(640, 640), overlap=0.2):
	height, width = image.shape[:2]
	win_h, win_w = window_size
	stride_h = int(win_h * (1 - overlap))
	stride_w = int(win_w * (1 - overlap))

	windows = {}

	# Generate all window positions
	y_positions = list(range(0, height - win_h + 1, stride_h))
	x_positions = list(range(0, width - win_w + 1, stride_w))

	# Ensure the far edges are covered by adding a final window at the edge
	if y_positions[-1] + win_h < height:
		y_positions.append(height - win_h)
	if x_positions[-1] + win_w < width:
		x_positions.append(width - win_w)

	# Slide windows across image
	for y_offset in y_positions:
		for x_offset in x_positions:
			window = image[y_offset:y_offset+win_h, x_offset:x_offset+win_w]
			window_norm = (window - window.min()) / (window.max() - window.min() + 1e-8) * 255
			windows[(y_offset, x_offset)] = window_norm

	predictions = model(list(windows.values()))
	df = pd.DataFrame()
	for window, prediction in zip(windows.keys(), predictions):
		y_offset, x_offset = window
		for box, conf in zip(prediction.boxes.xywh, prediction.boxes.conf):
			rectangle = yolo_box2rect(box)
			# Shift rectangle coordinates by window offset
			rectangle_shifted = [(y + y_offset, x + x_offset) for (y, x) in rectangle]
			_df = pd.DataFrame({
				'x0': [rectangle_shifted[0][1]],
				'y0': [rectangle_shifted[0][0]],
				'x1': [rectangle_shifted[2][1]],
				'y1': [rectangle_shifted[2][0]],
				'confidence': [conf.item()]
			})
			df = pd.concat([df, _df], ignore_index=True)

	return df

In [8]:
def extract_features(image: np.ndarray, boxes: pd.DataFrame) -> pd.DataFrame:
	"""Basic feature extraction using skimage.measure.regionprops_table."""

	features = pd.DataFrame()

	for _, box in boxes.iterrows():
		x0, y0, x1, y1 = int(box['x0']), int(box['y0']), int(box['x1']), int(box['y1'])
		crop = image[y0:y1, x0:x1]

		_features = pd.DataFrame(measure.regionprops_table(
			label_image=np.ones(crop.shape, dtype=int),  # Dummy label image
			intensity_image=crop,
			properties=[
				"area",
				"mean_intensity",
				"max_intensity",
				"min_intensity",
			],
		))
		_features["x0"] = x0
		_features["y0"] = y0
		_features["x1"] = x1
		_features["y1"] = y1
		_features["width"] = x1 - x0
		_features["height"] = y1 - y0
		if "confidence" in box:
			_features["confidence"] = box["confidence"]
		features = pd.concat([features, _features], ignore_index=True)
	# _label = draw.polygon2mask(projection.shape, [p0, p1, p2, p3]).astype(int)

	# label = label.squeeze(-1)  # Remove the channel axis if present



	# roi_feat_table = pd.DataFrame(measure.regionprops_table(
	# 		label_image=label,
	# 		intensity_image=image,
	# 		properties=[
	# 			"label",
	# 			"area",
	# 			"mean_intensity",
	# 			"max_intensity",
	# 			"min_intensity",
	# 		],
	# 	)
	# )
	
	# prediction = model(image)[0]
	# roi_feat_table = pd.DataFrame(columns=[
	# 	"x0", "y0", "x1", "y1",
	# 	"width", "height", "x_center", "y_center",
	# 	"area", "confidence"
	# ])

	# for idx, (box, conf) in enumerate(zip(prediction.boxes.xywh, prediction.boxes.conf)):
	# 	p0, p1, p2, p3 = yolo_box2bbox(box)
	# 	x_min, y_min = p0
	# 	x_max, y_max = p2

	# 	roi_feat_table.loc[idx, "x0"] = x_min
	# 	roi_feat_table.loc[idx, "y0"] = y_min
	# 	roi_feat_table.loc[idx, "x1"] = x_max
	# 	roi_feat_table.loc[idx, "y1"] = y_max
	# 	roi_feat_table.loc[idx, "width"] = x_max - x_min
	# 	roi_feat_table.loc[idx, "height"] = y_max - y_min
	# 	roi_feat_table.loc[idx, "x_center"] = (x_min + x_max) / 2
	# 	roi_feat_table.loc[idx, "y_center"] = (y_min + y_max) / 2
	# 	roi_feat_table.loc[idx, "area"] = (x_max - x_min) * (y_max - y_min)
	# 	roi_feat_table.loc[idx, "confidence"] = conf.detach().cpu().numpy()

	return features

In [9]:
def remove_duplicate_detections(df, iou_threshold=0.5):
	"""Remove duplicate detections in overlap regions using IoU."""
	if len(df) == 0:
		return df
	
	# Sort by confidence (highest first)
	df = df.sort_values('confidence', ascending=False).reset_index(drop=True)
	keep = []

	boxes_xyxy = df[['x0', 'y0', 'x1', 'y1']].values
	ious = box_iou(torch.tensor(boxes_xyxy), torch.tensor(boxes_xyxy)).numpy()
	np.fill_diagonal(ious, 0)  # Ignore self-comparison
	
	for i, row in df.iterrows():
		keep_this = True
		for kept_idx in keep:
			if ious[i, kept_idx] > iou_threshold:
				keep_this = False
				break
		
		if keep_this:
			keep.append(i)
	
	return df.loc[keep].reset_index(drop=True)

In [55]:
viewer.layers.clear()

for ms in scene.images:
	# retrieve translation and multiscales
	translation = [t for t in scene.metadata.coordinateTransformations if t.input.path == ms.name][0]
	translate = translation.translation[1:]
	scale=list(ms.images[0].scale.values())[1:]

	projection = ms.images[0].data.min(axis=0)
	projection = (projection - projection.min()) / (projection.max() - projection.min()) * 255


	df_rectangles = sliding_window_inference(projection.compute(), model, window_size=(640, 640), overlap=0.2)
	df_rectangles = remove_duplicate_detections(df_rectangles, iou_threshold=0.1)
	df_rectangles = extract_features(projection.compute(), df_rectangles)
	
	rectangles = []
	for _, row in df_rectangles.iterrows():
		x0 = float(row['x0']) * scale[1] + translate[1]
		y0 = float(row['y0']) * scale[0] + translate[0]
		x1 = float(row['x1']) * scale[1] + translate[1] 
		y1 = float(row['y1']) * scale[0] + translate[0]
		rectangles.append([(y0, x0), (y0, x1), (y1, x1), (y1, x0)])

	df_rectangles.drop(columns=["x0", "y0", "x1", "y1"], inplace=True)

	_ = viewer.add_image(projection, name=os.path.basename(ms.name), translate=translate, scale=scale)
	_ = viewer.add_shapes(
		rectangles, shape_type="rectangle", edge_color="red",
		face_color="transparent", features=df_rectangles, edge_width = 0.00001)



0: 640x640 (no detections), 17.8ms
1: 640x640 (no detections), 17.8ms
2: 640x640 (no detections), 17.8ms
3: 640x640 (no detections), 17.8ms
4: 640x640 (no detections), 17.8ms
5: 640x640 (no detections), 17.8ms
6: 640x640 (no detections), 17.8ms
7: 640x640 (no detections), 17.8ms
8: 640x640 (no detections), 17.8ms
9: 640x640 (no detections), 17.8ms
10: 640x640 (no detections), 17.8ms
11: 640x640 (no detections), 17.8ms
12: 640x640 (no detections), 17.8ms
13: 640x640 (no detections), 17.8ms
14: 640x640 (no detections), 17.8ms
15: 640x640 (no detections), 17.8ms
16: 640x640 (no detections), 17.8ms
17: 640x640 (no detections), 17.8ms
18: 640x640 (no detections), 17.8ms
19: 640x640 (no detections), 17.8ms
20: 640x640 (no detections), 17.8ms
21: 640x640 (no detections), 17.8ms
22: 640x640 (no detections), 17.8ms
23: 640x640 (no detections), 17.8ms
24: 640x640 (no detections), 17.8ms
25: 640x640 (no detections), 17.8ms
26: 640x640 (no detections), 17.8ms
27: 640x640 (no detections), 17.8ms
2

In [56]:
center_y = (y0 + y1) / 2 * scale[0] + translate[0]
center_x = (x0 + x1) / 2 * scale[1] + translate[1]
print(center_y, center_x)

0.04173144598976474 0.07976321026379112


In [64]:
layer = viewer.layers[1]

index = list(layer.selected_data)[0]

box = layer.data[index]
box_center = box.mean(axis=0)
box_center

array([   0.044702,    0.084524], dtype=float32)

In [65]:
stacklist = StackList(
        Entries=[
            StackEntry(
                Identifier="0170b408-1654-4929-dd9f-53ca3b71b923",
                Begin=2804.56 * 1e-6,
                End=2804.56 * 1e-6,
                SectionCount=1,
                ReferenceX=0.0465016000,
                ReferenceY=0.0255676000,
                FocusStabilizerOffset=0.0210407929,
                FocusStabilizerOffsetFixed=True,
                StackValid=True,
                Marked=True,
            ),
            # Add more entries as needed
        ],
    )

defined_regions = DefinedRegions(
        Entries=[
            DefinedRegionEntry(
                Identifier="7acd538d-f2e4-438a-caba-bcb44045f534",
                HolderRow=0,
                HolderColumn=0,
                CarrierIndex=0,
                ChamberRow=0,
                ChamberColumn=0,
                ChamberImageIndex=-1,
                MosaicRows=0,
                MosaicColumns=0,
            ),
            # Add more entries as needed
        ],
    )

col = CompoundShape(
    Name="9",
    Identifier="d9c8e1b0-5a3e-4f8b-9c7a-2f1b2c3d4e5f",
    Type="CompoundShape",
    Fill=[0, 1, 0, 0],
    Verticies=[],
    Children=[
        Point(
            Name="test",
            Tag="P2",
            Identifier="12345678-90ab-cdef-1234-567890abcdef",
            Type="Point",
            Fill=[0, 1, 0, 0],
            Verticies=[Vertex(X=box_center[1], Y=box_center[0])],
        ),
        # Add more Points or CompoundShapes as needed
    ],
)

row = CompoundShape(
    Name="E",
    Identifier="10fac5e0-dcaa-4138-77b7-32708f3b5bb6",
    Type="CompoundShape",
    Fill=[1, 0, 0, 0],
    Verticies=[],
    Children=[col]
)

shapelist = ShapeList(
        Items=[
            row,
        ]
	)

regions = Regions(ShapeList=shapelist)

In [66]:
stage_overview = StageOverviewRegions(
    Regions=regions,
	DefinedRegions=defined_regions,
	StackList=stacklist,
)

In [67]:
if os.path.exists("output.rgn"):
    os.remove("output.rgn")

xml_str = serialize_to_rgn(stage_overview)
# Parse and pretty-print with lxml, omitting XML declaration
xml_root = etree.fromstring(xml_str)
pretty_xml = etree.tostring(
    xml_root,
    pretty_print=True,
    encoding="utf-8",
    xml_declaration=False  # Omit the XML declaration
).decode("utf-8")

# Save to file
with open("output.rgn", "w") as f:
    f.write(pretty_xml)